# Proyecto 3: Galaxy Scaling Relations

Santiago Andrés Acosta Díaz

Disclaimer: los menús los hice ayudándome con IA

# Códigos para los menús


## Código para el menú de query

En esta parte, realizamos el código entero para el primer menú, que hace el query desde los datos de la NASA según el tipo de gráficas que deseemos hacer. Para ello, las funciones `action1` y `action2` son las encargadas de hacer el query y guardar el resultado del query en una base de datos de SQLite3.

Este primer menú sólo nos permite elegir los datos necesarios para hacer las gráficas correspondientes. Tiene una opción de filtro, la cual hace que el query sólo returne las filas en donde _todos los datos necesarios_ estén presentes.

### A considerar

1. Utilizamos `astroquery` para descargar la base de datos.
2. Le pasamos de una vez a `astroquery` una opción SQL para filtrar datos que no existan en todas las columnas que querramos. Esta condición se ve como `WHERE col1 IS NOT NULL AND col2 IS NOT NULL AND ...`
3. Lo anterior nos regresa una tabla de `astropy.table`. Para poder guardar el SQL `.db`, la convertimos a un `pandas.DataFrame` que directamente puede guardar la tabla SQL a través de una conexión temporal que creamos.

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive
import sqlite3
import pandas as pd

# Definimos los parámetros que, para cada gráfica, necesitamos 
checkbox_data = [
    ("Detection method distribution", ["discoverymethod"]),
    ("Period-radius diagram", ["pl_orbper", "pl_rade"]),
    ("Mass-radius relation", ["pl_bmasse", "pl_rade"]),   
    ("Equilibrium temperature histogram", ["pl_eqt"]),
    (r"$R_p$ by host star type", ["pl_rade", "st_spectype"]),   
    ("Discovery timeline", ["disc_year", "discoverymethod"]),   
]


# Esta es la lista en donde guardaremos las gráficas que vamos a hacer
graphs = []

# La base de datos como tal que resulta del astroquery
result = None

# Ahora, creamos las checkboxes para elegir las gráficas que queremos
#--------------------------------------------------------------------
# IA HELPED
# Create checkboxes
checkboxes = []
for label, string in checkbox_data:
    cb = widgets.Checkbox(
        value=False,
        description=label,
        indent=False
    )

    checkboxes.append((cb, string, label))

# Container to display the current list
output = widgets.Output()

unique_list = ["pl_name"]

# Function to rebuild the unique list from current checkbox states
def update_list(change=None):
    global unique_list, graphs

    graphs = []
    
    # Count active checkboxes per string
    active_counts = {}
    for cb, items, label in checkboxes:
        for string in items:
            if cb.value:
                active_counts[string] = active_counts.get(string, 0) + 1
        if cb.value:
            graphs.append(label)
    
    # Build the list: include a string only if its count > 0
    unique_list = ["pl_name"] + [s for s, count in active_counts.items() if count > 0]
    
    # Update the output area
    with output:
        clear_output(wait=True)
        print("Current query:", unique_list)

# Attach the update function to every checkbox
for cb, _, _ in checkboxes:
    cb.observe(update_list, names='value')

# Replace the "checkbox_grid" section with this:
row1 = widgets.HBox(
    [checkboxes[0][0], checkboxes[1][0], checkboxes[2][0]],
    layout=widgets.Layout(gap='30px')  # spacing between items
)
row2 = widgets.HBox(
    [checkboxes[3][0], checkboxes[4][0], checkboxes[5][0]],
    layout=widgets.Layout(gap='30px'))



# FILTER BUTTON

# Global variable to store the checkbox state
filterflag = False

# Create the checkbox
extra_checkbox = widgets.Checkbox(
    value=False,
    description="Apply filter",
    indent=False
)

# Observer to update the global variable
def update_global_flag(change):
    global filterflag
    filterflag = change['new']

extra_checkbox.observe(update_global_flag, names='value')


# BOTONES
button1 = widgets.Button(description="Query")
button2 = widgets.Button(description="Save Database")


# TEXTO DE GUARDADO
savepath = ""

# Create the text widget
text_input = widgets.Text(
    value='',
    placeholder='Guardar como...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='300px')   # optional width
)

# Observer: updates the global variable on every keystroke
def on_text_change(change):
    global savepath
    savepath = change['new']

text_input.observe(on_text_change, names='value')


# FUNCIONES DE LOS BOTONES
def action1(b):
    
    with output:
        global result
        
        clear_output(wait=True)
        print("Relizando query con parámetros:", unique_list)

        # Si queremos los datos filtrados, entonces corremos esto
        if filterflag:
            print("Filtrando filas NULL")

            # ### IMPORTANTE: aquí es donde le decimos a astroquery lo que queremos recibir. Como ella usa SQL, le podemos pedir que agregue la condición
            # que sólo nos devuelva rows que sí contengan todos los datos que queremos
            null_checks = " AND ".join([f"{col} IS NOT NULL" for col in unique_list])
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list, where=null_checks )

        # Esto si no los queremos filtrar
        else: 
            result = NasaExoplanetArchive.query_criteria(table="pscomppars", select=unique_list)

        # Esto es únicamente para comparar cuántos datos se filtran y así
        print(f"Resultados del query: {len(result)} resultados")


# Con este botón guardamos los datos en un SQL
def action2(b):
    with output:
        global savepath

        # Limpiamos el output del botón
        clear_output(wait=True)
        if savepath == "":
            print("Añadir nombre al archivo")
        else:
            if result == None:
                print("No hay datos")
            else:

                # Convertimos el Adtropy.table a un pandas.DataFrame
                df = result.to_pandas()
                
                # Spliteamos el nombre para asegurarnos que siempre guardamos un .db y no un .db.db
                names = savepath.split(".")

                # Nos conectamos a un SQL
                conn = sqlite3.connect(f'databases/{names[0]}.db')
                
                # Guardamos nuestra tabla en el SQL a través de la conexión
                df.to_sql('table', conn, if_exists='replace', index=False)
                
                # Cerramos la conección
                conn.close()

                print(f"Se guardó la tabla como {names[0]}.db")



#---------------------------------
# Creamos el menú
button1.on_click(action1)
button2.on_click(action2)

row3 = widgets.HBox(
    [extra_checkbox, button1, button2, text_input],
    layout=widgets.Layout(gap='30px')
)

ui = widgets.VBox([row1, row2])
ui_save = widgets.VBox([row3, output])

<string>:24: FutureWarning: tag.strict is not set. Currently defaults to False (permissive tag matching). In a future major version the default will change to True (require tags to contain a dot). Set tag.strict = true or tag.strict = false explicitly in your [tool.setuptools_scm] / [tool.vcs-versioning] config to silence this warning.


## Menú de carga de las bases de datos

Simplemente, leemos un archivo `.db` usando un cursor habiendo establecido una conexión a un servidos SQL. Sólo cerramos la conexión previa si volvemos a leer un nuevo archivo.

In [2]:

# Guardamos las variables de 
conn = None     # La conexión al servidor SQL
cursor = None   # El cursos global de la conexión
loadpath = ""   # EL path a la base de datos a usar

# Variables para saber el contenido de nuestras bases
columns_load = list()
type_load = list()


#--------------------------------
# IA HELPED TO CREATE THE MENU

# Container to display the current list
output_load = widgets.Output()

# Create the text widget
text_input_load = widgets.Text(
    value='',
    placeholder='Cargar archivo...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='300px')   # optional width
)

# Observer: updates the global variable on every keystroke
def on_text_change_load(change):
    global loadpath
    loadpath = change['new']

text_input_load.observe(on_text_change_load, names='value')


# Botón de carga
button_load = widgets.Button(description="Load Database")

# Lista de columnas en la base de datos
column_list = list()

# Cargamos la base de datos
def action2_load(b):
    # Estas variables son globales para no matarme pasando variables a funciones
    global loadpath, conn, cursor, columns_load, type_load
    with output_load:

        # Esto lo hacemos en caso tal que abramos otra base de datos: si nuestro cursos existe, entonces ya hemos
        # Abierto una conexión. Si cargamos esta función, es porque queremos crear una conexión nueva
        if cursor is not None:
            conn.close()

        clear_output(wait=True)   # Se limpia el output del botón

        # Esto es básicamente robado del botón de save arriba
        if loadpath == "":
            print("Añadir nombre al archivo")
        else:
            # El try, si detecta un error, nos bota el error en vez de simplemente matar la ejecución
            try:
                names = loadpath.split(".")   # Para siempre cargar .db

                # Creamos la conexión y el cursor. El check_same_thread es un truquito para evitar errores en otras celdas
                conn = sqlite3.connect(f'file:databases/{names[0]}.db?mode=rw', uri=True, check_same_thread=False)
                cursor = conn.cursor()
                
                print(f"Se cargó la tabla {names[0]}.db")   # Aviso

                # Esto lo hacemos para asegurarnos que la base de datos no esté vacía. Si lo está, bota error. 
                # Si no lo está, nos devuelve una lista de info de las columnas
                cursor.execute(f"PRAGMA table_info('table')")
                columns = cursor.fetchall()

                # Reiniciamos los datos info para el filtrado posterior
                columns_load = list()
                type_load = list()

                # Aquí, revisamos el nombre de la columna y el tipo para luego hacer el filtro especializado
                for data in columns:
                    columns_load.append(data[1])
                    type_load.append(data[2])

                # Para saber qué datos contiene la tabla
                print(f"Con datos de {columns_load}")

            # Si algo arriba falla, entonces la base dada no se puede leer
            except sqlite3.DatabaseError as e:
                print(f"No se puede acceder a la base de datos: {e}")
            except sqlite3.OperationalError as e:
                print(f"No se puede acceder a la base de datos: {e}")


# Creamos el botón y todo el container
button_load.on_click(action2_load)

row_load = widgets.HBox(
    [button_load, text_input_load],
    layout=widgets.Layout(gap='30px')
)

ui_load = widgets.VBox([row_load, output_load])

## Código del menú para los filtros personalizados

Donde, por cada tabla cargada, se obtienen sus posibles valores y se aplican los filtros según corresponda con una query de SQL. 

Este se encarga de poblar la query inicial `SELECT * FROM 'table'` según las columnas contenidas en la gráfica. Para datos numéricos, agrega condiciones `WHERE x IS BETWEEN x_0 AND x_1`. Para datos de texto que contengan opciones, puebla usando `WHERE x IN (...)`.

### También se usó SQL para

1. Para las columnas tipo texto, obtenemos todas las posibles opciones usando SQL con `SELECT DISTINC FROM columna FROM tabla`
2. Para los datos de espectrometría, hacemos lo mismo de arriba pero obtenemos sólo la primera letra del tipo de técnica usando `SELECT DISTINCT SUBSTR(columna, 1, 1) FROM tabla`
3. Para los datos numéricos, obtenemos los valores mínimo y máximos (con otra condición para eliminar un valor muy atípico en el año de descubrimiento).

In [3]:
QUERY_PREFIX = "SELECT * FROM 'table'"   
conditions = {}                         
full_query_output = widgets.Output()


# Query global que se va a modificar
QUERY = ""


def rebuild_query():
    # En esta función reescribimos toda la QUERY
    global QUERY

    # Query global que se va a modificar
    QUERY = ""
    
    with full_query_output:
        full_query_output.clear_output()

        # Las conditions son, como tal, las condiciones que vamos a generar por cada columna.
        active = [cond for cond in conditions.values()]

        # Poblamos el prefix con las condiciones que tenemos
        if active:
            query = QUERY_PREFIX + "\nWHERE " + "AND ".join(active)
        else:
            query = QUERY_PREFIX

        # Guardamos el QUERY global
        QUERY = query
        print(query)   # Hacemos el display 


# Esta función hace el check de las opciones que cada columna de tipo texto ofrece
def get_distinct_values(column_name):

    # Nombre general de la tabla
    table_name = 'table'

    # Revisamos que en efecto hallamos cargado algo
    if cursor is None:
        print("No database loaded.")
        return []
    try:

        # En el caso del tipo de espectroscopía, nos interesa únicamente la primera letra 
        if column_name == "st_spectype": 
            cursor.execute(f"SELECT DISTINCT SUBSTR({column_name}, 1, 1) FROM '{table_name}' ORDER BY 1")
            return [row[0] for row in cursor.fetchall()[1::]]

        # En el otro tipo, sí podemos robarnos todo
        elif column_name == "discoverymethod":
            cursor.execute(f"SELECT DISTINCT {column_name} FROM '{table_name}' ORDER BY 1")
            return [row[0] for row in cursor.fetchall()]

    # En caso de haber error, decimos que la base de datos está maluca
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return []


# Con esto preparamos los valores para ser displayados en los sliders de rango
def get_min_max(column_name):

    # Lo mismo de arriba
    table_name = 'table'
    if cursor is None:
        print("No database loaded.")
        return None, None

        
    try:
        # Seleccionamos el mínimo y el máximo de las columnas numéricas. En este caso, hay un dato con `discovery_year` = 0 que
        # aparentemente no tiene sentido, entonces nos aseguramos de quitar ese también
        cursor.execute(f"""
            SELECT
                (SELECT MIN({column_name})
                 FROM '{table_name}'
                 WHERE {column_name} > 0) AS min_positive,
                (SELECT MAX({column_name})
                 FROM '{table_name}') AS real_max
        """)

        # Hacemos el fetch del query
        row = cursor.fetchone()

        # Y revisamos que no bote error
        if row and row[0] is not None and row[1] is not None:
            return row[0], row[1]
        else:
            return None, None

    # En caso de que bote error, entonces avisamos
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return None, None
    
        

# Esta es la función encargada de realizar los filtros específicos de cada tabla
def create_filters():

    # conditions es la lista en donde agregamos los `WHERE ...` de cada columna
    global conditions 

    conditions = {}    # La vaciamos

    # Revisamos el nombre y el tipo de cada columna
    for name, coltype in zip(columns_load, type_load):

        # Simplemente no hacemos nada con el nombre del planeta.
        if name == "pl_name":
            continue

        # En las columnas de texto, queremos coger todas las posibles opciones
        if coltype == "TEXT":

            # TITLE
            print(f"\t\t\t\t\t\t {name.upper()} \n\n")

            # Obtenemos las posibles opciones y creamos checkboxes 
            options = get_distinct_values(name)
            checkboxes = [widgets.Checkbox(description=opt, value=True) for opt in options]
            
            # Las organizamos en filitas de a 4 para que se vea bonito
            chunk_size = 4
            chunks = [checkboxes[i:i+chunk_size] for i in range(0, len(checkboxes), chunk_size)]
            checkbox_rows = [widgets.HBox(chunk) for chunk in chunks]
            checkbox_vbox = widgets.VBox(checkbox_rows)

            # Cuadro de texto 
            output = widgets.Output()
    
            # Con esto creamos la query SQL
            def update_string(change, cb_list=checkboxes, out=output, col = name):
                with out:
                    out.clear_output()
                    selected = [cb.description for cb in cb_list if cb.value]

                    # IA suggestion
                    quoted = [f"'{opt.replace("'", "''")}'" for opt in selected]

                    if name == "st_spectype":
                        snippet = f"SUBSTR({col}, 1, 1) IN ({', '.join(quoted)})\n"
                    else:
                        snippet = f"{col} IN ({', '.join(quoted)})\n"

                    # Guardamos el `WHERE...` en la lista de condiciones
                    conditions[col] = snippet           # store for the full query
                    rebuild_query() 
                    
            # Cada checkbox va a enlazarse con la respectiva función
            for cb in checkboxes:
                cb.observe(update_string, names='value')
    
            # Estado inicial
            update_string(None)
    
            # Mostraomos el menú de una vez
            display(checkbox_vbox, output)
            print("\n\n\n")
            
        else:
            # En el caso numérico, creamos los sliders dado el respectivo rango
            min_val, max_val = get_min_max(name)
            
            # TITLE
            print(f"\t\t\t\t\t\t {name.upper()} \n\n")

            
            if min_val is not None and max_val is not None:
                if coltype == "INTEGER":
                    # Las columnas enteras no necesitan de logaritmización
                    slider = widgets.IntRangeSlider(
                        value=[min_val, max_val],
                        min=min_val,
                        max=max_val,
                        step=1,
                        description=name,
                        layout=widgets.Layout(width='80%'),
                        continuous_update=False
                    )
                    is_log = False
                
                else:  
                    # Algunas columnas float sí necesitan logaritmización a excepción de pl_rade y pl_eqt
                    if name in ["pl_rade", "pl_eqt"]:
                        slider = widgets.FloatRangeSlider(
                            value=[min_val, max_val],
                            min=min_val,
                            max=max_val,
                            step=0.01,
                            description=name,
                            layout=widgets.Layout(width='80%'),
                            continuous_update=False
                        )
                        is_log = False
                        
                    else:

                        # En los otros casos, sí se necesita logaritmizar
                        min_exp = np.log10(min_val)
                        max_exp = np.log10(max_val)
                        init_low = np.log10(min_val)
                        init_high = np.log10(max_val)
            
                        slider = widgets.FloatRangeSlider(
                            value=[init_low, init_high],
                            min=min_exp,
                            max=max_exp,
                            step=0.01,         
                            description=name,
                            layout=widgets.Layout(width='80%'),
                            continuous_update=False,
                            readout_format='.2f' 
                        )
                        is_log = True
                
                # Cuadro de output
                output = widgets.Output()
                
                # Aquí es donde se codifica lo que tiene el slider
                def on_range_change(change, out=output, col=name, log_flag=is_log):
                    with out:
                        out.clear_output()
                        low, high = change['new']

                        # Si es un slider logarítmico, entonces hacemos la respectiva conversión
                        if log_flag:
                            actual_low = 10 ** low
                            actual_high = 10 ** high
                            print(f"Actual range: {actual_low:.4f} to {actual_high:.4f}")
                        else:
                            # Si es uno normal, entonces no pasa nada
                            actual_low = low
                            actual_high = high
                
                        # Construimos el quey usando el valor real (sea log o no)
                        snippet = f"{col} BETWEEN {actual_low} AND {actual_high}\n"
                        conditions[col] = snippet
                        rebuild_query()   # Llamamos a reconstruir toda la query desde el comienxo
                
                # Enlazamos slider con la función
                slider.observe(on_range_change, names='value')
                
                # Valores iniciales
                if is_log:
                    init_change = {'new': [np.log10(min_val), np.log10(max_val)]}
                else:
                    init_change = {'new': [min_val, max_val]}
                
                on_range_change(init_change)
                
                # Hacemos el display
                row_widget = widgets.VBox([slider, output])
                display(row_widget)
                print("\n\n\n")

    rebuild_query()  # Actualizamos la query

    print("\n\n \t\t\t QUERY FOR THE MOMENT")
    display(full_query_output) # Mostramos el menú entero

## Código del menú para las gráficas

In [4]:
# Container to display the current list
output_read = widgets.Output()
df = None

def check_values(COLS, data_dict):
    
    cols_set = set(COLS)  # Convert to set for O(1) membership tests
    result = []

    for key, value_list in data_dict.items():
        # Check if every item in value_list exists in cols_set
        if all(item in cols_set for item in value_list):
            result.append(key)

    return result


# Para realizar la QUERY, hacer checks y llamar a las gráficas
def read_data(b):
    
    
    with output_read:

        global graphs, df
        
        output_read.clear_output()
        # Se realiza la SQL query
        df = pd.read_sql_query(QUERY, conn)
    
        # Miramos los datos que tiene
        COLS = list(df.keys())
    
        # Revisamos que las gráficas que se quieren hacer se puedan hacer con las gráficas que tenemos
        possible_graphs = check_values(COLS, dict(checkbox_data))

        flag = all(g in possible_graphs for g in graphs)

        # whatafac why do I have to negate this?????
        if not flag:
            print("ERROR: you don't have enough data to make the graphs")
            print(f"You can make the following graphs")
            print(possible_graphs)
        else: 
            print("Data read. Graphs can be done")

    
# Create the button
button_read_data = widgets.Button(
    description="Read and check data",
    button_style="primary",  # 'primary', 'success', 'info', 'warning', 'danger' or ''
    tooltip="Realiza la Query de SQL y la guarda en un pd.DataFrame. Revisa que los datos tengan las columnas necesarias para hacer las gráficas que se piden",
    layout=widgets.Layout(width='300px')  # Let the button size itself
)

# Attach the function to the button's on_click event
button_read_data.on_click(read_data)

# Center the button using a container box
container = widgets.VBox([button_read_data, output_read])

## Funciones para hacer las gráficas

Aquí es donde está la física en realidad. En esta parte, por simplicidad (y para hacer que lo que hice arriba tuviera sentido), seleccioné e hice los cálculos para cada gráfica a partir del pandas.DataFrame del SQL query. Para ver los resultados usando puro SQL, creé un script de python simple que los tiene. Se nos piden 5 gráficas:

1. Distribución del método de detección en un histograma: solo hacemos un count the cada método `discovery_method` y graficamos el histograma.
2. Diagrama periodo-radio: graficamos el periodo `pl_orber` contra el radio `pl_rade`.
3. Histograma de temperatura de equilibrio (por zonas): cogemos `pl_eqt` y hacemos el histograma en bins (por default). Pero seleccionamos diferentes zonas (fría $<200$K, templada $<800$K, caliente $>800$K) y las coloreamos respectivamente.
4. Box-Plots del radio del planeta respecto al tipo de estrella host: box-plot de `pl_rade` contra `sl_spectrype`.
5. Timeline de descubrimiento por año y método: Hacemos un histograma cumulativo del count de planetas descubiertos separando el método de descubrimiento.

__NOTA:__ la interpretación física se va a mostrar luego al momento de mostrar los resultados. 

In [25]:
# Otros imports para hacer las cosas bonitas
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import textwrap

# output de las gráficas
out_plots = widgets.Output()

# Usamos seaborn para que quere bonito
sns.set_theme()
sns.set_style('whitegrid')
sns.set_context('notebook')

pal_name = 'PuOr'
sns.set_palette(pal_name)  # Una paleta que me gusta


# Distribución de método de detección: histograma 
def plot_method_distribution(ax, data):
    
    # Hacemos el count del método de detección de los datos
    counts = data['discoverymethod'].value_counts()

    palette = sns.color_palette(pal_name, len(counts))
    
    # Creamos el histograma
    ax.bar(counts.index, counts.values, color = palette)
    ax.set_title('Detection Method Distribution')

    # Lo modificamos
    ax.set_xlabel('Method')
    ax.set_ylabel('Count')

    # Workaround para labels muy largas
    def wrap_labels(labels, max_len=12):
        return ['\n'.join(textwrap.wrap(label, max_len)) for label in labels]
    ax.set_xticklabels(wrap_labels(counts.index, max_len=12), rotation=45, ha='right')


# 2. Diagrama periodo vs radio
def plot_period_radius(ax, data):

    # Vamos a crear unos colorcitos lindos
    colors = np.log10(data['pl_orbper']) * data['pl_rade']
    
    # Graficamos los datos
    ax.scatter(data['pl_orbper'], data['pl_rade'], c = colors, s=8, alpha=0.6)
    
    # Usamos escala logarítmica
    ax.set_xscale('log')
    ax.set_yscale('log')

    # Añadimos labels y title
    ax.set_title('Period–Radius Diagram')
    ax.set_xlabel('Orbital Period (days)')
    ax.set_ylabel('Planet Radius (R$_E$)')

    
# 3. Relación masa-radio
def plot_mass_radius(ax, data):
    
    # Vamos a crear unos colorcitos lindos
    colors = np.log10(data['pl_bmasse']) * np.log10(data['pl_rade'])
    
    # Hacemos el plot de los datos
    ax.scatter(data['pl_bmasse'], data['pl_rade'], c = colors, s=8, alpha=0.6)

    # Ponemos escala logarítmica
    ax.set_xscale('log')
    ax.set_yscale('log')

    # -- Aquí hacemos la regresión de los datos

    # Linealizamos nuestros datos 
    log_m = np.log10(data['pl_bmasse'])
    log_r = np.log10(data['pl_rade'])

    # Hacemos el fit
    coeffs = np.polyfit(log_m, log_r, 1)

    # Hacemos la predicción
    x_fit = np.linspace(log_m.min(), log_m.max(), 100)
    y_fit = np.polyval(coeffs, x_fit)

    # Graficamos la regresión (convertir a log)
    ax.plot(10**x_fit, 10**y_fit, color='red', lw=2,
            label=f'R ∝ M$^{{{coeffs[0]:.2f}}}$')

    # Labels y títulos
    ax.legend()
    ax.set_title('Mass–Radius Relation')
    ax.set_xlabel('Planet Mass (M$_E$)')
    ax.set_ylabel('Planet Radius (R$_E$)')

    
# Temperatura de equilibrio por zonas
def plot_temp_histogram(ax, data):
    
    # Definimos planetas en las distintas zonas
    cold = data[data['pl_eqt'] < 200]
    temperate = data[(data['pl_eqt'] >= 200) & (data['pl_eqt'] <= 800)]
    hot = data[data['pl_eqt'] > 800]

    # -- Creamos los histogramas por separado junto con suus respectivos colores
    ax.hist(cold['pl_eqt'], bins=15, alpha=0.6, label='Cold (<200 K)', color='blue')
    ax.hist(temperate['pl_eqt'], bins=15, alpha=0.6, label='Temperate (200–800 K)', color='orange')
    ax.hist(hot['pl_eqt'], bins=15, alpha=0.6, label='Hot (>800 K)', color='red')

    # Labels y títulos
    ax.set_title('Equilibrium Temperature by Zone')
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('Count')
    ax.legend()

    
# 5. Boxplots del radio planetario respecto a la estrella host
def plot_rp_by_spectype(ax, data):
    
    # Extraemos primero el tipo de estrella host
    data['spec_class'] = data['st_spectype'].str[0].str.upper()

    # Hacemos el box-plot
    sns.boxplot(ax=ax, x='spec_class', y='pl_rade', data=data, palette=pal_name)

    # Labels y title
    ax.set_title('Planet Radius by Host Star Type')
    ax.set_xlabel('Spectral Class')
    ax.set_ylabel('Planet Radius (R$_E$)')


    
# 6. Conteo cumulativo por método de descubrimiento
def plot_discovery_timeline(ax, data):
    
    # Agrupamos los datos por año de descubrimiento y por método 
    grouped = data.groupby(['disc_year', 'discoverymethod']).size().unstack(fill_value=0)
                                                            # Contamos cada uno de estos
    # Suma comulativa
    cum = grouped.cumsum()

    # Ordenamos de mayor a menor cumsum para evitar overlap
    sorted_columns = cum.iloc[-1].sort_values().index 
    cum_sorted = cum[sorted_columns]

    # Creamos la paleta
    palette = sns.color_palette(pal_name, len(cum_sorted.index))
    
    # Hacemos el plot con los datos ordenados
    ax.stackplot(cum_sorted.index, cum_sorted.T, labels=cum_sorted.columns, alpha=1., cmap = pal_name)
    ax.set_title('Discovery Timeline (Cumulative)')
    ax.set_xlabel('Discovery Year')
    ax.set_ylabel('Cumulative Count')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize='x-small')



# ----------------------------------------------------------------------
# Orchestrator: builds a figure from a list of plot names
# ----------------------------------------------------------------------
def create_exoplanet_figure(trash):
    global df, graphs

    with out_plots:
        out_plots.clear_output()
        data = df 
    
        plot_list = graphs
        
        # Map plot names to their functions
        func_map = {
            'Detection method distribution': plot_method_distribution,
            'Period-radius diagram': plot_period_radius,
            'Mass-radius relation': plot_mass_radius,
            'Equilibrium temperature histogram': plot_temp_histogram,
            '$R_p$ by host star type': plot_rp_by_spectype,
            'Discovery timeline': plot_discovery_timeline,
        }
    
        n = len(plot_list)
    
        # Arrange subplots in a grid with 2 columns (except when only 1 plot)
        if n == 1:
            rows, cols = 1, 1
        else:
            cols = 2
            rows = (n + 1) // 2

        fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), constrained_layout=True)

        if n == 1:
            ax = axes
            axes = [ax]
        else:
            axes = axes.flatten()

        
        for i, plot_name in enumerate(plot_list):
            if i >= len(axes):
                break
            func = func_map.get(plot_name)
            if func is None:
                axes[i].text(0.5, 0.5, f'Unknown: {plot_name}', ha='center', va='center')
                axes[i].set_axis_off()
            else:
                func(axes[i], data)
    
        # Turn off any unused subplots
        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)
        
        plt.tight_layout(pad=3.0)
        # plt.subplots_adjust(left=0.08, right=0.92, bottom=0.08, top=0.92, wspace=0.25, hspace=0.35)
        plt.savefig("plots.pdf")
        plt.show()



# Crear botón
button_plot = widgets.Button(
    description="Make plots",
    button_style="primary", 
    layout=widgets.Layout(width='300px')  
)

# Enlazar botón al orquestrador
button_plot.on_click(create_exoplanet_figure)

# Crear su contenedor
container_plot = widgets.VBox([button_plot, out_plots])



# Menú



## Elegir las gráficas que se van a realizar

Aquí elegimos, a priori, las columnas que vamos a utilizar en el query

In [6]:
display(ui)
display(ui_save)

## Menú de carga

Aquí cargamos la base de datos que vamos a utilizar

In [7]:
display(ui_load)

## Menú de filtros específicos

Por cada una de las tablas que tengamos, vamos a crear unos filtros especiales que le funcionen sólo y únicamente a esas tablas. 
En el caso de columnas de tipo texto, vamos a tener un menú donde podemos elegir qué opciones mantener y cuáles no. 
Para la columna `spec_type`, las opciones serán únicamente la primera letra del tipo de emisión.
Para datos numéricos, se tiene un slider con un rango.

In [11]:
create_filters()

						 DISCOVERYMETHOD 




Output()





						 PL_ORBPER 








						 PL_RADE 








						 PL_BMASSE 




...





						 ST_SPECTYPE 




Output()





						 DISC_YEAR 










 			 QUERY FOR THE MOMENT


Output()

## Menú para hacer las gráficas

Básicamente me robo el primer menú para mostrar las gráficas a hacer. Si no se pueden hacer las gráficas con el database seleccionado, entonces se dice error.

In [9]:
display(ui)
display(container)

In [26]:
display(container_plot)